# UFC Strike Tracker — Exploratory Data Analysis

This notebook explores the scraped UFCStats data to understand strike distributions, 
fighter tendencies, and patterns useful for the detection pipeline.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RAW_DIR = Path('../data/raw')

In [ ]:
# Load all scraped fights
records = []
for p in RAW_DIR.glob('*.json'):
    fight = json.loads(p.read_text())
    for rnd in fight.get('rounds', []):
        for fi in range(2):
            sig_raw = rnd.get(f'fighter_{fi}_sig_str', '0 of 0').split(' of ')
            tot_raw = rnd.get(f'fighter_{fi}_total_str', '0 of 0').split(' of ')
            records.append({
                'fight_id': fight['fight_id'],
                'fighter': fight['fighters'][fi] if fi < len(fight.get('fighters', [])) else f'fighter_{fi}',
                'round': rnd['round'],
                'sig_landed': int(sig_raw[0]) if sig_raw[0].isdigit() else 0,
                'sig_attempted': int(sig_raw[1]) if len(sig_raw) > 1 and sig_raw[1].isdigit() else 0,
                'total_landed': int(tot_raw[0]) if tot_raw[0].isdigit() else 0,
                'total_attempted': int(tot_raw[1]) if len(tot_raw) > 1 and tot_raw[1].isdigit() else 0,
            })

df = pd.DataFrame(records)
print(f'{len(df)} round-fighter rows from {df["fight_id"].nunique()} fights')
df.head()

In [ ]:
# Sig strike accuracy distribution
df['accuracy'] = df['sig_landed'] / df['sig_attempted'].replace(0, np.nan)
df['accuracy'].dropna().hist(bins=30)
plt.xlabel('Sig strike accuracy')
plt.ylabel('Count')
plt.title('Distribution of per-round significant strike accuracy')
plt.show()

In [ ]:
# Strikes per round by round number
df.groupby('round')[['sig_landed', 'total_landed']].mean().plot(kind='bar')
plt.title('Average strikes landed per round')
plt.ylabel('Strikes')
plt.xticks(rotation=0)
plt.show()